### Data Ingestion

In [ ]:
# from langchain_community.document_loaders import RecursiveUrlLoader

# import re
# from bs4 import BeautifulSoup

# def bs4_extractor(html: str) -> str:
#     soup = BeautifulSoup(html, "html.parser")
#     # Extract the main content of the page, for example, by getting the text from the body tag)
#      # remove noise
#     for tag in soup(["script", "style", "header", "footer", "nav"]):
#         tag.decompose()
    
#     # preserve table structure

#     for table in soup.find_all("table"):
#         rows = []
#         for row in table.find_all("tr"):
#             cells = [c.get_text(strip=True) for c in row.find_all(["td", "th"])]
#             rows.append(" | ".join(cells))
#         table.replace_with("\n".join(rows))
    
#     return re.sub(r"\n\n+", "\n\n", soup.get_text(separator="\n", strip=True)).strip()

# loader = RecursiveUrlLoader("https://reference.langchain.com/python/langchain-community/document_loaders/recursive_url_loader/RecursiveUrlLoader", max_depth=2, use_async=False, timeout=5, extractor=bs4_extractor, base_url="https://reference.langchain.com/python/langchain-community/document_loaders/recursive_url_loader/RecursiveUrlLoader")

# documents = []
# document_lazy = loader.lazy_load()

# for doc in document_lazy:
#     documents.append(doc)
# print(documents[0].page_content[:500])  # Print the first 500 characters of the first document
# print(f"Total documents loaded: {len(documents)}")
# print(documents[0].metadata)

In [ ]:
from langchain_community.document_loaders import FireCrawlLoader
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env file

firecrawl_api_key = os.getenv("FIRECRAWL_API_KEY")

loader = FireCrawlLoader(
api_key=firecrawl_api_key,
url="https://reference.langchain.com/python/langchain-community/document_loaders/recursive_url_loader/RecursiveUrlLoader", 
mode = "crawl", params = {
    "limit": 10,
    "allowBackwardCrawling": False,
    "scrapeOptions": {
        "waitfor": 2000
    }
})

docs = []
docs_lazy = loader.lazy_load()

# async variant:
# docs_lazy = await loader.alazy_load()

for doc in docs_lazy:
    docs.append(doc)
    
print(docs[0].page_content[:500]) # Print the first 500 characters of the first document
print(docs[0].metadata)
print(f"Total documents loaded: {len(docs)}")

### Chunking

In [ ]:
### text splitting get into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents,chunk_size = 1000,chunk_overlamp=200):
    """Split documents into smaller chunks for better processing and embedding generation."""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlamp,
    length_function = len,
    separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of the chunk
    if split_docs:
        print("Example chunk:")
        print(f"content: {split_docs[0].page_content[:500]}")  # print the first 500 characters of the first chunk
        print(f"metadata: {split_docs[0].metadata}")
    return split_docs

In [ ]:
chunks = split_documents(docs)

### Embedding and VectorStoreDB

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer # embedding model is in sentence_transformers library
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

d:\Github repo\RAG-Powered-Website-Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """Manages the creation and storage of embeddings for documents."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

       """Initialize the embedding manager
       
       Args:
              model_name: Hugging Face model name for generating embeddings
       """
       self.model_name = model_name
       self.model = None # initially set to None, will be loaded when needed to initialize the model
       self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimension: {self.model.get_embedding_dimension()}") # print the dimension of the embeddings every text will have the same 384 dimension for this model
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
           
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts using the loaded model.
        
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initiialize the embedding manager and generate embeddings for the chunks
embedding_manager = EmbeddingManager()
embedding_manager

### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in chromadb vector store."""

    def __init__(self, collection_name: str = "documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store and create a collection for storing document embeddings.
        
        Args:
            collection_name: Name of the collection to store document embeddings
            persist_directory: Directory where the vector store will be persisted
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()
    
    def _initialize_vector_store(self):
         """Initialize the chromadb client and collection to storing document embeddings."""
         try:
            # create persist chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path= self.persist_directory)

            # create or get collection for storing document embeddings
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "Collection for storing document embeddings for RAG"})
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
         except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their corresponding embeddings to the vector store.
        
        Args:
            documents: List of document objects (e.g., langchain Document instances)
            embeddings: numpy array of embeddings corresponding to the documents, shape (len(documents), embedding_dim)
        """
        if not self.collection:
            raise ValueError("Vector store collection is not initialized.")
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store...")
       
        #prepare data for chromadb
        ids = []
        metadatas = []
        document_texts = []
        embeddings_list = []
    
        for i,(doc, embedding) in enumerate(zip(documents, embeddings)):
           # Generate unique ID
           doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
           ids.append(doc_id)

           # Prepare metadata (you can customize this based on your document structure)
           metadata = dict(doc.metadata)
           metadata["doc_index"] = i
           metadata['content_length'] = len(doc.page_content)
           metadatas.append(metadata)

           # Document content
           document_texts.append(doc.page_content)

           # Embeddings
           embeddings_list.append(embedding.tolist())

         # Add to chromadb collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store